In [0]:
df = spark.read \
    .option("inferSchema", "true") \
    .option("header", "true") \
    .csv("/Volumes/second_data_engineering_project/landing/raw_files/olist_order_reviews_dataset.csv").printSchema()

In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField("review_id", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("review_score", IntegerType(), True),
    StructField("review_comment_title", StringType(), True),
    StructField("review_comment_message", StringType(), True),
    StructField("review_creation_date", TimestampType(), True),
    StructField("review_answer_timestamp", TimestampType(), True)
])
df = spark.read \
    .schema(schema) \
    .option("header", "true") \
    .csv("/Volumes/second_data_engineering_project/landing/raw_files/olist_order_reviews_dataset.csv")
df.createOrReplaceTempView("raw_reviews")

In [0]:
df.display()

In [0]:
from pyspark.sql import functions as F
df.filter(
    ~F.col("review_id").rlike("^[0-9a-fA-f]{32}$")
    ).count()

In [0]:
df.filter(
    ~F.col("order_id").rlike("^[0-9a-fA-f]{32}$")
    ).count()

In [0]:
df_quotes = spark.read \
    .schema(schema) \
    .option("header", "true") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("unescapedQuoteHandling", "STOP_AT_CLOSING_QUOTE") \
    .csv("/Volumes/second_data_engineering_project/landing/raw_files/olist_order_reviews_dataset.csv")

In [0]:
df_quotes.filter(
    ~F.col("review_id").rlike("^[0-9a-fA-F]{32}$")
).count()

In [0]:
df_quotes = spark.read \
    .schema(schema) \
    .option("header", "true") \
    .option("sep", ",") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("unescapedQuoteHandling", "STOP_AT_CLOSING_QUOTE") \
    .csv("/Volumes/second_data_engineering_project/landing/raw_files/olist_order_reviews_dataset.csv")

In [0]:
df_quotes.filter(
    ~F.col("review_id").rlike("^[0-9a-fA-F]{32}$")
).count()

In [0]:
df_quotes.filter(
    ~F.col("order_id").rlike("^[0-9a-fA-F]{32}$")
).count()


In [0]:
from pyspark.sql import functions as F

raw_reviews = spark.read.text(
    "/Volumes/second_data_engineering_project/landing/raw_files/olist_order_reviews_dataset.csv"
)

In [0]:
hex_id = r"[0-9a-fA-F]{32}"
timestamp = r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}"
score = r"[1-5]"

In [0]:
pattern = (
    rf"^({hex_id}),"
    rf"({hex_id}),"
    rf"({score}),"
    rf"(.*),"
    rf"({timestamp}),"
    rf"({timestamp})$"
)

In [0]:
df_reconstructed = raw_reviews.select(
    "value",

    F.regexp_extract(
        F.col("value"), pattern, 1
    ).alias("review_id"),

    F.regexp_extract(
        F.col("value"), pattern, 2
    ).alias("order_id"),

    F.regexp_extract(
        F.col("value"), pattern, 3
    ).alias("review_score"),

    F.regexp_extract(
        F.col("value"), pattern, 4
    ).alias("review_text"),

    F.regexp_extract(
        F.col("value"), pattern, 5
    ).alias("review_creation_date"),

    F.regexp_extract(
        F.col("value"), pattern, 6
    ).alias("review_answer_timestamp")
)

In [0]:
df_valid_structure = df_reconstructed.filter(
    (F.col("review_id") != "") &
    (F.col("order_id") != "") &
    (F.col("review_score") != "") &
    (F.col("review_creation_date") != "") &
    (F.col("review_answer_timestamp") != "")
)

df_valid_structure.count()

In [0]:
raw_reviews.count()

In [0]:
df_valid_structure.display(truncate=False)

In [0]:
df_invalid_structure = df_reconstructed.filter(
    (F.col("review_id") == "") |
    (F.col("order_id") == "") |
    (F.col("review_score") == "") |
    (F.col("review_creation_date") == "") |
    (F.col("review_answer_timestamp") == "")
)

df_invalid_structure.display(truncate=False)

df_invalid_structure.count()

In [0]:
raw_reviews = spark.read.text(
    "/Volumes/second_data_engineering_project/landing/raw_files/olist_order_reviews_dataset.csv"
).show(truncate=False)

In [0]:
hex_id = r"[0-9a-fA-F]{32}"
timestamp = r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}"
score = r"[1-5]"

pattern = (
    rf"^.*?({hex_id})"
    rf".*?({hex_id})"
    rf".*?({score})"
    rf"(.*)"
    rf"({timestamp})"
    rf".*?"
    rf"({timestamp})$"
)

In [0]:
df_reconstructed = raw_reviews.select(
    "value",

    F.regexp_extract(F.col("value"), pattern, 1).alias("review_id"),
    F.regexp_extract(F.col("value"), pattern, 2).alias("order_id"),
    F.regexp_extract(F.col("value"), pattern, 3).alias("review_score"),
    F.regexp_extract(F.col("value"), pattern, 4).alias("comments"),
    F.regexp_extract(F.col("value"), pattern, 5).alias("review_creation_date"),
    F.regexp_extract(F.col("value"), pattern, 6).alias("review_answer_timestamp")
)

In [0]:
df_valid_structure = df_reconstructed.filter(
    (F.col("review_id") != "") &
    (F.col("order_id") != "") &
    (F.col("review_score") != "") &
    (F.col("comments") != "") &
    (F.col("review_creation_date") != "") &
    (F.col("review_answer_timestamp") != "")
)

df_valid_structure.count()

In [0]:
df_valid_structure.display(truncate=False)

In [0]:
raw_reviews.count()

In [0]:
raw_reviews.count() - df_valid_structure.count()

In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField("review_id", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("review_score", IntegerType(), True),
    StructField("review_comment_title", StringType(), True),
    StructField("review_comment_message", StringType(), True),
    StructField("review_creation_date", TimestampType(), True),
    StructField("review_answer_timestamp", TimestampType(), True)
])

df_multiline_reviews = spark.read \
    .schema(schema) \
    .option("header", "true") \
    .option("multiLine", "true") \
    .option("escape", '"') \
    .csv("/Volumes/second_data_engineering_project/landing/raw_files/olist_order_reviews_dataset.csv")

df_multiline_reviews.createOrReplaceTempView("raw_reviews_correct")

In [0]:
df_multiline_reviews.display()

In [0]:
df_multiline_reviews.filter(
    ~F.col("review_id").rlike("^[0-9a-fA-F]{32}$")
).count()

In [0]:
df_multiline_reviews.filter(
    ~F.col("order_id").rlike("^[0-9a-fA-F]{32}$")
).count()

In [0]:
df_orders = spark.read \
    .option("inferSchema", "true") \
    .option("header", "true") \
    .csv("/Volumes/second_data_engineering_project/landing/raw_files/olist_orders_dataset.csv")

In [0]:
df_multiline_reviews.join(
    df_orders,
    on="order_id",
    how="left_anti"
).count()

In [0]:
%sql
SELECT * FROM text.`/Volumes/second_data_engineering_project/landing/raw_files/olist_order_reviews_dataset.csv`
LIMIT 20